### Cell 1: Import Required Libraries
We start by importing the necessary tools from the libraries we just installed.

In [1]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import evaluate
import numpy as np

### Cell 2: Data Acquisition
The goal is to fine-tune a model for sentiment analysis (binary classification). We will load the public IMDb Movie Review Dataset as instructed.

In [2]:
# Load the IMDb dataset
dataset = load_dataset("imdb")

# For faster training during a demonstration, we can optionally create a smaller subset
# small_train_dataset = dataset["train"].shuffle(seed=42).select(range(2000))
# small_test_dataset = dataset["test"].shuffle(seed=42).select(range(500))

print(dataset["train"][0]) # Take a look at the data

{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

### Cell 3: Load the Tokenizer
We need to load a pre-trained model checkpoint suitable for classification. We will use `distilbert-base-uncased` and load its corresponding Tokenizer.

In [3]:
model_checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

### Cell 4: Data Preprocessing (Tokenization)
We must use the loaded tokenizer to convert the raw text data into the numerical inputs (tokens and attention masks) required by the model

In [4]:
def preprocess_function(examples):
    # Truncate to ensure texts fit the model's maximum input length
    return tokenizer(examples["text"], truncation=True, padding="max_length")

# Apply the tokenization to all splits in the dataset
tokenized_datasets = dataset.map(preprocess_function, batched=True)

### Cell 5: Load the Pre-Trained Model
 We load the pre-trained `distilbert-base-uncased` model using the `AutoModelForSequenceClassification` class, specifying 2 labels for our binary classification task (positive/negative).

In [5]:
# Load the pre-trained model for sequence classification
model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=2)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### Cell 6: Define Evaluation Metrics
The project requires evaluating the model and reporting metrics like Accuracy and F1-Score. We will set up a function to compute these.

In [6]:
metric_accuracy = evaluate.load("accuracy")
metric_f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    
    accuracy = metric_accuracy.compute(predictions=predictions, references=labels)["accuracy"]
    f1 = metric_f1.compute(predictions=predictions, references=labels)["f1"]
    
    return {"accuracy": accuracy, "f1": f1}

### Cell 7: Fine-Tuning Setup and Execution
We use the Hugging Face Trainer API to fine-tune the model on the preprocessed training data. The instructions specify fine-tuning for 2-3 epochs.

In [7]:
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2, # Fine-tuning for 2 epochs as per instructions
    weight_decay=0.01,
    eval_strategy="epoch", 
    save_strategy="epoch",
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"], 
    eval_dataset=tokenized_datasets["test"],   
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

# Start fine-tuning
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.226777,0.197733,0.923200,0.921395
2,0.151850,0.235612,0.930760,0.931061


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=3126, training_loss=0.20664039240841367, metrics={'train_runtime': 1322.9042, 'train_samples_per_second': 37.796, 'train_steps_per_second': 2.363, 'total_flos': 6623369932800000.0, 'train_loss': 0.20664039240841367, 'epoch': 2.0})

### Cell 8: Final Evaluation
Now we evaluate the newly fine-tuned model on the test set to report the final Accuracy and F1-Score.

In [8]:
# Evaluate the model on the test dataset
evaluation_results = trainer.evaluate()

print("Final Evaluation Results:")
for key, value in evaluation_results.items():
    print(f"{key}: {value:.4f}")

Final Evaluation Results:
eval_loss: 0.1977
eval_accuracy: 0.9232
eval_f1: 0.9214
eval_runtime: 157.9244
eval_samples_per_second: 158.3040
eval_steps_per_second: 9.8970
epoch: 2.0000


### Cell 9: Inference on Unseen Data
Finally, we implement a function to take a new, unseen sentence and predict its sentiment.

In [9]:
def predict_sentiment(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True).to(model.device)
    with torch.no_grad():
        logits = model(**inputs).logits
    
    predicted_class_id = logits.argmax().item()
    labels = ["Negative", "Positive"]
    return labels[predicted_class_id]

# Test the inference function
new_review_1 = "The pacing was a bit slow, but the cinematography was absolutely breathtaking!"
new_review_2 = "I honestly couldn't wait for this movie to end. Terrible acting."

print(f"Review 1 Sentiment: {predict_sentiment(new_review_1)}")
print(f"Review 2 Sentiment: {predict_sentiment(new_review_2)}")

Review 1 Sentiment: Positive
Review 2 Sentiment: Negative


### Project Summary & Key Concepts

**Transfer Learning in this Project:**
Transfer learning is applied here by taking a model (DistilBERT) that has already learned vast amounts of knowledge from general text corpora, and fine-tuning it on a much smaller, specific dataset (IMDb movie reviews). Instead of training a model from scratch—which requires massive compute and data—we leverage the pre-trained model's existing understanding of grammar, syntax, and language patterns, and simply adjust its final layers to understand the specific task of sentiment analysis. 

**Power of Pre-trained LLMs:**
* Highly accurate even with relatively small task-specific datasets.
* Saves immense amounts of computational resources and time.
* Excellent out-of-the-box generalization capabilities.

**Limitations of Pre-trained LLMs:**
* They can be large and memory-intensive to run (though DistilBERT mitigates this).
* They can inherit biases present in their original massive training data.
* They operate as "black boxes," making it difficult to precisely explain *why* they make a specific classification.

**Final Conclusion & Metrics:**
The fine-tuning process was highly successful. After just two epochs of training on the IMDb dataset, the DistilBERT model achieved an **Accuracy of 92.32%** and an **F1-Score of 92.14%**. This demonstrates how effectively transfer learning allows a pre-trained LLM to adapt to a specific classification task with minimal training time.